# LITE Results File Availability and Integrity Check

Checks the compact result bundles under `I001_Results/LITE/SIM_NNN/`. Each complete bundle contains:

- `mode0.npz` — smallest stiffness eigenmode history
- `curves.npz` — force/displacement, shear, and force-chain summaries
- `geometry.npz` — deforming coordinates and mesh connectivity
- `metadata.json` — simulation and lattice metadata

The notebook checks file presence and size, opens every file without pickle support, validates required keys and array shapes, checks finite numeric values and monotonic time axes, and compares the time axes shared by the curve and geometry products. The default range is SIM 5140–5149.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 220)

# Work whether Jupyter starts in the repository root or this notebook's folder.
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'I001_Results').is_dir() else CWD.parent
RESULTS_DIR = ROOT / 'I001_Results'
LITE_DIR = RESULTS_DIR / 'LITE'

SIM_START = 5140
SIM_END = 5149
SIMS = list(range(SIM_START, SIM_END + 1))
EXPECTED_CURVE_FRAMES = 401  # Set to None to disable this completeness check.
EXPECTED_MODE_FRAMES = 401   # Set to None to disable this completeness check.

FILES = ('mode0.npz', 'curves.npz', 'geometry.npz', 'metadata.json')
MIN_SIZE_BYTES = {
    'mode0.npz': 100_000,
    'curves.npz': 1_000,
    'geometry.npz': 100_000,
    'metadata.json': 1_000,
}

print(f'Repository: {ROOT}')
print(f'LITE results: {LITE_DIR}')
print(f'Simulations: {SIM_START}-{SIM_END}')

## Validation rules

In [ ]:
REQUIRED_KEYS = {
    'mode0.npz': {
        't', 'matrix_index', 'eigenvalue', 'eigenvector', 'dof_labels',
    },
    'curves.npz': {
        't', 'u2', 'rf2', 'shear_mean',
        'global_ef_t', 'global_ef_c',
        'global_ef_t_allnodes', 'global_ef_c_allnodes', 'n_nodes_total',
    },
    'geometry.npz': {
        't', 'node_ids', 'coordinates', 'reference_node_labels',
        'reference_coordinates', 'elements', 'element_lengths',
        'element_types', 'hole_boundary_nodes', 'hole_boundary_lengths',
    },
}

def _check_time(name, values, errors):
    values = np.asarray(values)
    if values.ndim != 1 or values.size == 0:
        errors.append(f'{name}: t must be a non-empty 1-D array')
        return
    if not np.isfinite(values).all():
        errors.append(f'{name}: t contains NaN or infinity')
    if values.size > 1 and np.any(np.diff(values) < 0):
        errors.append(f'{name}: t is not monotonic')

def _check_finite(name, values, errors):
    values = np.asarray(values)
    if np.issubdtype(values.dtype, np.number) and not np.isfinite(values).all():
        errors.append(f'{name} contains NaN or infinity')

def validate_npz(path, filename):
    errors = []
    details = {}
    try:
        with np.load(path, allow_pickle=False) as data:
            keys = set(data.files)
            missing = sorted(REQUIRED_KEYS[filename] - keys)
            if missing:
                errors.append('missing keys: ' + ', '.join(missing))
                return errors, details

            t = data['t']
            _check_time(filename, t, errors)
            n_t = len(t) if t.ndim == 1 else 0
            details['frames'] = n_t
            details['t_min'] = float(t[0]) if n_t else np.nan
            details['t_max'] = float(t[-1]) if n_t else np.nan

            if filename == 'curves.npz':
                series = [
                    'u2', 'rf2', 'shear_mean', 'global_ef_t', 'global_ef_c',
                    'global_ef_t_allnodes', 'global_ef_c_allnodes',
                ]
                for key in series:
                    values = data[key]
                    if values.shape != (n_t,):
                        errors.append(f'{key} shape {values.shape} != ({n_t},)')
                    _check_finite(key, values, errors)
                if np.asarray(data['n_nodes_total']).size != 1:
                    errors.append('n_nodes_total is not scalar')
                details['n_nodes_total'] = int(np.asarray(data['n_nodes_total']).reshape(-1)[0])

            elif filename == 'geometry.npz':
                coordinates = data['coordinates']
                node_ids = data['node_ids']
                reference = data['reference_coordinates']
                elements = data['elements']
                element_lengths = data['element_lengths']
                if coordinates.ndim != 3 or coordinates.shape[-1] != 2:
                    errors.append(f'coordinates has invalid shape {coordinates.shape}')
                elif coordinates.shape[:2] != (n_t, len(node_ids)):
                    errors.append('coordinates does not match t and node_ids')
                if reference.shape != (len(node_ids), 2):
                    errors.append('reference_coordinates does not match node_ids')
                if elements.ndim != 2 or len(element_lengths) != len(elements):
                    errors.append('elements and element_lengths are inconsistent')
                if len(data['element_types']) != len(elements):
                    errors.append('element_types does not match elements')
                if len(data['reference_node_labels']) != len(node_ids):
                    errors.append('reference_node_labels does not match node_ids')
                if len(data['hole_boundary_lengths']) != len(data['hole_boundary_nodes']):
                    errors.append('hole boundary arrays are inconsistent')
                _check_finite('coordinates', coordinates, errors)
                _check_finite('reference_coordinates', reference, errors)
                details['nodes'] = len(node_ids)
                details['elements'] = len(elements)
                details['holes'] = len(data['hole_boundary_nodes'])

            elif filename == 'mode0.npz':
                eigenvalue = data['eigenvalue']
                eigenvector = data['eigenvector']
                matrix_index = data['matrix_index']
                dof_labels = data['dof_labels']
                if eigenvalue.shape != (n_t,):
                    errors.append(f'eigenvalue shape {eigenvalue.shape} != ({n_t},)')
                if matrix_index.shape != (n_t,):
                    errors.append(f'matrix_index shape {matrix_index.shape} != ({n_t},)')
                if eigenvector.ndim != 2 or eigenvector.shape[0] != n_t:
                    errors.append('eigenvector does not match t')
                if dof_labels.ndim != 2 or dof_labels.shape[1] != 2:
                    errors.append(f'dof_labels has invalid shape {dof_labels.shape}')
                elif eigenvector.ndim == 2 and eigenvector.shape[1] != len(dof_labels):
                    errors.append('eigenvector width does not match dof_labels')
                _check_finite('eigenvalue', eigenvalue, errors)
                _check_finite('eigenvector', eigenvector, errors)
                details['dofs'] = len(dof_labels)
    except Exception as exc:
        errors.append(f'{type(exc).__name__}: {exc}')
    return errors, details

def validate_metadata(path, sim):
    errors = []
    details = {}
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('format') != 'FOAM_VIDEO_3200_LITE_V1':
            errors.append(f"unexpected format: {payload.get('format')!r}")
        if payload.get('simulation') != sim:
            errors.append(f"simulation is {payload.get('simulation')!r}, expected {sim}")
        lattice = payload.get('lattice')
        if not isinstance(lattice, dict):
            errors.append('lattice metadata is missing')
        else:
            details.update({
                'nodes_metadata': lattice.get('number_of_nodes'),
                'elements_metadata': lattice.get('number_of_elements'),
                'holes_metadata': lattice.get('number_of_holes'),
                'mesh_kind': lattice.get('mesh_kind'),
            })
    except Exception as exc:
        errors.append(f'{type(exc).__name__}: {exc}')
    return errors, details

## Scan and validate all bundles

In [ ]:
file_rows = []
bundle_details = {}

for sim in SIMS:
    sim_dir = LITE_DIR / f'SIM_{sim:03d}'
    bundle_details[sim] = {}
    for filename in FILES:
        path = sim_dir / filename
        exists = path.is_file()
        size = path.stat().st_size if exists else None
        errors = []
        details = {}
        if exists and size < MIN_SIZE_BYTES[filename]:
            errors.append(f'suspiciously small ({size:,} bytes)')
        if exists:
            validation_errors, details = (
                validate_metadata(path, sim)
                if filename == 'metadata.json'
                else validate_npz(path, filename)
            )
            errors.extend(validation_errors)
        bundle_details[sim][filename] = details
        file_rows.append({
            'SIM': sim,
            'file': filename,
            'exists': exists,
            'size_MB': size / (1024 ** 2) if size is not None else np.nan,
            'status': 'missing' if not exists else ('valid' if not errors else 'invalid'),
            'details': '; '.join(errors),
            **details,
        })

files_df = pd.DataFrame(file_rows)
files_df

## Availability and integrity table

Green means the file exists and passed its checks, red means it exists but failed at least one check, and grey means it is missing. Values are file sizes in MB.

In [ ]:
size_table = files_df.pivot(index='SIM', columns='file', values='size_MB').reindex(columns=FILES)
status_table = files_df.pivot(index='SIM', columns='file', values='status').reindex(columns=FILES)

STATUS_COLORS = {
    'valid': 'background-color:#c6efce; color:#006100',
    'invalid': 'background-color:#ffc7ce; color:#9c0006',
    'missing': 'background-color:#f2f2f2; color:#999999',
}

def color_status(_):
    return pd.DataFrame(
        [[STATUS_COLORS[status_table.loc[sim, filename]] for filename in size_table.columns]
         for sim in size_table.index],
        index=size_table.index, columns=size_table.columns,
    )

(size_table.style
 .apply(color_status, axis=None)
 .format(lambda value: '-' if pd.isna(value) else f'{value:,.2f}')
 .set_caption('LITE file size (MB), colored by validation status'))

## Per-simulation summary and cross-file checks

In [ ]:
summary_rows = []
for sim in SIMS:
    subset = files_df[files_df['SIM'] == sim].set_index('file')
    problems = []
    for filename in FILES:
        row = subset.loc[filename]
        if row['status'] != 'valid':
            explanation = row['details'] or row['status']
            problems.append(f'{filename}: {explanation}')

    curves = bundle_details[sim].get('curves.npz', {})
    geometry = bundle_details[sim].get('geometry.npz', {})
    metadata = bundle_details[sim].get('metadata.json', {})
    if curves and EXPECTED_CURVE_FRAMES is not None and curves.get('frames') != EXPECTED_CURVE_FRAMES:
        problems.append(f"curve history has {curves.get('frames')} frames; expected {EXPECTED_CURVE_FRAMES}")
    mode = bundle_details[sim].get('mode0.npz', {})
    if mode and EXPECTED_MODE_FRAMES is not None and mode.get('frames') != EXPECTED_MODE_FRAMES:
        problems.append(f"eigenmode history has {mode.get('frames')} frames; expected {EXPECTED_MODE_FRAMES}")
    if curves and geometry:
        curves_path = LITE_DIR / f'SIM_{sim:03d}' / 'curves.npz'
        geometry_path = LITE_DIR / f'SIM_{sim:03d}' / 'geometry.npz'
        try:
            with np.load(curves_path, allow_pickle=False) as c, np.load(geometry_path, allow_pickle=False) as g:
                if not np.array_equal(c['t'], g['t']):
                    problems.append('curves/geometry time axes differ')
        except Exception:
            pass  # The individual validator already reports unreadable files.
    if geometry and metadata:
        for observed_key, metadata_key, label in [
            ('nodes', 'nodes_metadata', 'node count'),
            ('elements', 'elements_metadata', 'element count'),
            ('holes', 'holes_metadata', 'hole count'),
        ]:
            if metadata.get(metadata_key) is not None and geometry.get(observed_key) != metadata.get(metadata_key):
                problems.append(f'{label} differs between geometry and metadata')

    summary_rows.append({
        'SIM': sim,
        'complete': not problems,
        'valid_files': int((subset['status'] == 'valid').sum()),
        'curve_frames': curves.get('frames'),
        'mode_frames': mode.get('frames'),
        'nodes': geometry.get('nodes'),
        'elements': geometry.get('elements'),
        'problems': '; '.join(problems),
    })

summary_df = pd.DataFrame(summary_rows).set_index('SIM')
print(f"Complete LITE bundles: {int(summary_df['complete'].sum())} / {len(summary_df)}")
summary_df

## Show only problems

In [ ]:
problem_files_df = files_df[files_df['status'] != 'valid'][
    ['SIM', 'file', 'status', 'size_MB', 'details']
]
problem_files_df if len(problem_files_df) else 'No file-level problems found.'

## Quick visual inspection

Change `SIM_TO_PLOT` to inspect another complete bundle.

In [ ]:
SIM_TO_PLOT = 5145
sim_dir = LITE_DIR / f'SIM_{SIM_TO_PLOT:03d}'

with np.load(sim_dir / 'curves.npz', allow_pickle=False) as curves, \
     np.load(sim_dir / 'mode0.npz', allow_pickle=False) as mode:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    axes[0, 0].plot(curves['u2'], curves['rf2'])
    axes[0, 0].set(xlabel='U2', ylabel='RF2', title='Force–displacement')

    axes[0, 1].plot(curves['t'], curves['shear_mean'])
    axes[0, 1].set(xlabel='time', ylabel='shear_mean', title='Mean shear measure')

    axes[1, 0].plot(curves['t'], curves['global_ef_t'], label='tension')
    axes[1, 0].plot(curves['t'], curves['global_ef_c'], label='compression')
    axes[1, 0].set(xlabel='time', ylabel='effective fraction', title='Force-chain fractions')
    axes[1, 0].legend()

    axes[1, 1].plot(mode['t'], mode['eigenvalue'])
    axes[1, 1].axhline(0, color='black', linewidth=0.8)
    axes[1, 1].set(xlabel='time', ylabel='smallest eigenvalue', title='Smallest stiffness eigenvalue')

    fig.suptitle(f'SIM {SIM_TO_PLOT} — LITE result overview', fontsize=14)
plt.show()

## Metadata for one simulation

In [ ]:
metadata_path = LITE_DIR / f'SIM_{SIM_TO_PLOT:03d}' / 'metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
metadata['lattice']

## Notes

- Edit `SIM_START` and `SIM_END` in the configuration cell to check another range.
- The default expected history length is 401 frames. Change `EXPECTED_CURVE_FRAMES` and `EXPECTED_MODE_FRAMES`, or set either to `None`, for workflows with a different schedule.
- A present file is marked invalid if it is suspiciously small, cannot be opened, lacks required arrays, contains non-finite values, or has inconsistent shapes.
- `curves.npz` and `geometry.npz` must have identical time axes. `mode0.npz` may use an independent sampling schedule, so it is validated separately.
- The source `RES_SIM_NNN_LITE.csv.gz` is an intermediate extraction file and may be deleted after a successful reduction; it is not required for a complete final LITE bundle.